# RAG: Baseline vs Re-Ranking — IFAB *Laws of the Game 2024/25*

Two retrieval pipelines over the **same chunks** of one long PDF:

1. **Baseline** — Bi-Encoder, top 10 by cosine similarity, top 3 go to the LLM.
2. **Re-ranking** — the same top 10, re-scored by a Cross-Encoder, top 3 go to the LLM.

**In plain words.** The PDF is cut into small pieces called *chunks*. When we ask a question, a fast model
(the Bi-Encoder) finds the 10 chunks that look closest to the question. The baseline hands the best 3 to the
LLM. The second pipeline first lets a slower, more careful model (the Cross-Encoder) re-order those same 10,
and hands over *its* best 3. Then we compare the two answers.

| Word | Meaning here |
|---|---|
| Chunk | A short piece of the PDF (at most 160 words) that the system retrieves as one unit |
| Token | A small piece of a word, the unit that language models read. "offside" may be 2 tokens |
| Bi-Encoder | Turns the question and each chunk into number lists (vectors) separately, then compares them |
| Cross-Encoder | Reads the question and one chunk together and gives one relevance score |
| Gold chunk | The chunk that really contains the answer, chosen by hand for each test question |
| Rank | Position in a list: rank 1 is the best. Only ranks 1 to 3 reach the LLM |

The notebook is split into stages. Only what is slow or hand-made is saved to disk; the cheap steps are
recomputed on every run, so "Restart and run all" always works.

| Stage | Saved to disk |
|---|---|
| Stage 0 — setup | nothing (reads `data/config.json`) |
| Stage 1 — corpus | nothing (rebuilt from the PDF each run) |
| Stage 2 — chunks | `data/chunks.jsonl`, frozen so chunk ids stay stable |
| Stage 3 — embeddings and index | nothing (recomputed, about a second) |
| Stage 4–5 — retrieval and re-ranking | nothing (recomputed) |
| Stage 6 — queries and rank shifts | `data/queries.json` (hand-made), `results/rank_shifts.csv` |
| Stage 6b — Recall@3 and MRR | `results/retrieval_metrics.csv` |
| Stage 7 — Answer 1 and Answer 2 | `results/answers.json` (so a re-run makes no API calls) |
| Stage 8 — latency | `results/latency.csv` |

## Stage 0 — Setup

Load the settings from `data/config.json` once. Every later stage reads the same values, so the
models, chunk sizes and top-k are never typed in two places.

In [1]:
# Imports and settings. Everything the notebook needs to know (model names, chunk sizes, top-k) is
# read once from data/config.json, so no number is typed in two places.
import json, re, sys, random
from pathlib import Path
from collections import OrderedDict

from pypdf import PdfReader

project_root = Path.cwd()                                  # this notebook sits in repo/, so the current folder is repo/
config = json.loads((project_root / "data" / "config.json").read_text())
pdf_path = project_root / config["source_pdf"]
random.seed(config["seed"])

print("python :", sys.executable)
print("PDF    :", pdf_path.name, "| found:", pdf_path.exists())
print("include:", len(config["include_sections"]), "sections")

python : /home/noel/Desktop/Obsidian/MTech_CSE/SEM3/Subjects/LMA/Assignments/RAG-Reranking/repo/.venv/bin/python
PDF    : Laws-of-the-Game-2024_25.pdf | found: True
include: 20 sections


## Stage 1 — Corpus: PDF to clean paragraphs

The PDF is a booklet: most PDF pages are **two printed pages side by side**. The text extractor gives the
left page first, then the right one, so reading order is fine. What needs cleaning:

- a **footer** on every page (`Laws of the Game 2024/25 | Law 11 | Offside`). It is useful: it tells us
  which Law the page belongs to, so we read it for a label and then remove it from the text;
- the **printed page numbers** glued together at the top (e.g. `3938`);
- a vertical **side tab** (`Law`, `11`, `Offside`) that comes out as three stray lines;
- **line breaks** in the middle of sentences. Paragraph starts have a leading space, bullets start
  with `•`, and headings look like `2. Offside offence`, so paragraphs can be rebuilt from those clues.

We keep only the sections listed in `include_sections`. The contents page, the "About the Laws"
introduction, the glossary and the "what changed" summary are left out. The glossary and the change list
repeat rule text, which would give near-identical chunks.

In [2]:
# The footer printed on every page, e.g. "Laws of the Game 2024/25   |  Law 1  |  The Field of Play".
# We read it to learn which Law the page belongs to, then remove it from the text.
FOOTER_PATTERN = re.compile(r"Laws of the Game 2024/25\s*\|\s*(.+?)\s*$")

# The official title of each Law. Used only to spot and remove the vertical side tab on each page.
LAW_TITLES = {
    1: "The Field of Play", 2: "The Ball", 3: "The Players", 4: "The Players’ Equipment",
    5: "The Referee", 6: "The Other Match Officials", 7: "The Duration of the Match",
    8: "The Start and Restart of Play", 9: "The Ball in and out of Play",
    10: "Determining the Outcome of a Match", 11: "Offside", 12: "Fouls and Misconduct",
    13: "Free Kicks", 14: "The Penalty Kick", 15: "The Throw-in", 16: "The Goal Kick",
    17: "The Corner Kick",
}

def read_pages(pdf_file):
    """Read the PDF page by page. Each page is tagged with its section (taken from the footer)."""
    reader = PdfReader(str(pdf_file))
    pages_read = []
    current_section = "front matter"                      # a page with no footer inherits the previous page's section
    for page_number, page in enumerate(reader.pages, start=1):
        kept_lines, footer_labels = [], []
        for line in (page.extract_text() or "").split("\n"):
            footer_match = FOOTER_PATTERN.search(line)
            if footer_match:
                # "Law 1  |  The Field of Play" -> keep only "Law 1"
                footer_labels.append(footer_match.group(1).split("|")[0].strip())
                line = FOOTER_PATTERN.sub("", line)       # then cut the footer out of the text
            kept_lines.append(line.rstrip())
        if footer_labels:
            current_section = footer_labels[-1]
        # The printed page numbers sit at the very top (e.g. "3938"); drop that first line.
        if kept_lines and re.fullmatch(r"\s*\d{1,6}\s*", kept_lines[0]):
            kept_lines = kept_lines[1:]
        pages_read.append({"pdf_page": page_number, "section": current_section, "lines": kept_lines})
    return pages_read

def drop_side_tab(lines, section_name):
    """Remove the vertical tab on the page edge. It comes out of the extractor as 'Law' / '<number>' / '<title words>'."""
    law_match = re.fullmatch(r"Law (\d+)", section_name)
    if not law_match:
        return lines                                       # only the Law pages have a side tab
    law_title = LAW_TITLES[int(law_match.group(1))]
    cleaned_lines, position = [], 0
    while position < len(lines):
        is_tab_start = (lines[position].strip() == "Law" and position + 1 < len(lines)
                        and re.fullmatch(r"\s*\d+\s*", lines[position + 1]))
        if is_tab_start:
            position += 2                                  # skip "Law" and the number
            # skip the title words that follow, as long as they really are part of the title
            while position < len(lines) and lines[position].strip() and lines[position].strip() in law_title:
                position += 1
            continue
        cleaned_lines.append(lines[position])
        position += 1
    return cleaned_lines

In [3]:
BULLET_START = re.compile(r"^[\s ]*•")         # a line that starts with a bullet point
NUMBERED_HEADING = re.compile(r"^\s*\d{1,2}\.\s+\S")      # a line like "2. Offside offence"

def tidy(text):
    """Collapse repeated spaces and fix '( excluding the halfway line )' into '(excluding the halfway line)'."""
    text = re.sub(r"\s+", " ", text.replace(" ", " ")).strip()
    text = re.sub(r"\(\s+", "(", text)
    return re.sub(r"\s+\)", ")", text)

def build_paragraphs(page_lines):
    """Join (pdf_page, line) pairs back into whole paragraphs and headings.
    The PDF breaks lines in the middle of sentences, so we look at how each line STARTS to know
    whether it begins something new or just continues the previous line."""
    paragraphs, current = [], None

    def finish_current():
        """Store the paragraph we have been building (if any) and start fresh."""
        nonlocal current
        if current and tidy(current["text"]):
            paragraphs.append({"kind": current["kind"], "text": tidy(current["text"]), "pdf_page": current["pdf_page"]})
        current = None

    for page_number, line in page_lines:
        if not line.strip():
            continue
        plain_line = line.replace(" ", " ")
        # A heading is short, starts with "2.", and does not end like a sentence.
        is_heading = (bool(NUMBERED_HEADING.match(plain_line)) and len(plain_line.split()) <= 8
                      and not plain_line.rstrip().endswith((".", ":", ";", ",")))
        if is_heading:                                     # a heading stands on its own
            finish_current()
            paragraphs.append({"kind": "heading", "text": tidy(plain_line), "pdf_page": page_number})
        elif plain_line.startswith(" ") or BULLET_START.match(line) or NUMBERED_HEADING.match(plain_line):
            finish_current()                               # a new paragraph or bullet starts on this line
            current = {"kind": "para", "text": plain_line, "pdf_page": page_number}
        elif current is None:
            current = {"kind": "para", "text": plain_line, "pdf_page": page_number}
        else:
            current["text"] += " " + plain_line            # same paragraph, just wrapped onto a new line
    finish_current()
    return paragraphs

In [4]:
all_pages = read_pages(pdf_path)

# Keep only the sections we want, and glue each section's pages together in order, so a paragraph
# that runs over a page break stays in one piece.
lines_by_section = OrderedDict()
for page in all_pages:
    if page["section"] in config["include_sections"]:
        page_lines = drop_side_tab(page["lines"], page["section"])
        lines_by_section.setdefault(page["section"], []).extend((page["pdf_page"], line) for line in page_lines)
corpus = {section_name: build_paragraphs(lines) for section_name, lines in lines_by_section.items()}

# Two kinds of noise survive the steps above, so we remove them and PRINT what was removed:
#  - the big page title at the start of each Law (e.g. "The Players"), which just repeats the Law's name;
#  - labels from diagrams (e.g. the signal graphics: "CORNER KICK GOAL KICK OFFSIDE ..."), which get glued
#    onto the END of a real paragraph. We cut the labels off and keep the rule that came before them.
KNOWN_ACRONYMS = {"VAR", "AVAR", "IFAB", "FIFA"}           # capital-letter words that are real, not diagram labels

def is_shouting(word):
    """True for an ALL-CAPS word of 3+ letters that is not a known acronym."""
    return bool(re.fullmatch(r"[A-Z]{3,}", word)) and word not in KNOWN_ACRONYMS

def clean_noise(paragraph, section_name):
    """Return (cleaned paragraph or None, a note saying what was done). Cuts the text at the
    first two ALL-CAPS words in a row, because that is where diagram labels begin."""
    text = paragraph["text"]
    law_match = re.fullmatch(r"Law (\d+)", section_name)
    is_page_title = (law_match and paragraph["kind"] == "para" and len(text.split()) <= 8
                     and text.strip() in LAW_TITLES[int(law_match.group(1))])
    if is_page_title:
        return None, "page title"
    if paragraph["kind"] != "para":
        return paragraph, None                             # headings are left alone
    words = text.split()
    for i in range(len(words) - 1):
        if is_shouting(words[i].strip(".,;:()")) and is_shouting(words[i + 1].strip(".,;:()")):
            text_before_labels = " ".join(words[:i])
            if len(text_before_labels.split()) < 6 or text_before_labels.startswith("Refer to graphics"):
                return None, "diagram labels only"
            return {**paragraph, "text": text_before_labels}, "labels cut: " + " ".join(words[i:i + 6]) + " ..."
    return paragraph, None

removed_paragraphs = []
for section_name in corpus:
    kept_paragraphs = []
    for paragraph in corpus[section_name]:
        cleaned_paragraph, note = clean_noise(paragraph, section_name)
        if note:
            removed_paragraphs.append((section_name, note, paragraph["text"]))
        if cleaned_paragraph:
            kept_paragraphs.append(cleaned_paragraph)
    corpus[section_name] = kept_paragraphs
print(f"noise handled in {len(removed_paragraphs)} paragraphs:")
for section_name, note, text in removed_paragraphs:
    print(f"  [{section_name} | {note}]")
    print("      was:", text[:100])
print()

count_words = lambda text: len(text.split())

# A summary table: how much text each section ended up with.
print(f"{'section':<42}{'pdf pages':>10}{'paragraphs':>12}{'words':>8}")
for section_name, paragraphs in corpus.items():
    pdf_pages_used = sorted({paragraph["pdf_page"] for paragraph in paragraphs})
    total_words = sum(count_words(paragraph["text"]) for paragraph in paragraphs)
    print(f"{section_name:<42}{str(pdf_pages_used[0]) + '-' + str(pdf_pages_used[-1]):>10}{len(paragraphs):>12}{total_words:>8}")

pages_left_out = OrderedDict()
for page in all_pages:
    if page["section"] not in config["include_sections"]:
        pages_left_out[page["section"]] = pages_left_out.get(page["section"], 0) + 1
print("\nleft out (section: pdf pages):", dict(pages_left_out))

noise handled in 8 paragraphs:
  [Law 3 | page title]
      was: The Players
  [Law 5 | labels cut: FREE KICK Advantage (2) FREE KICK ...]
      was: Referees and other ‘on-field ’ match officials are prohibited from wearing jewellery or any other el
  [Law 5 | diagram labels only]
      was: Refer to graphics for approved referee signals. Indirect free kick FREE KICK FREE KICK Corner kick C
  [Law 5 | labels cut: CORNER KICK GOAL KICK OFFSIDE PENALTY ...]
      was: • any other decision taken in accordance with the Laws of the Game or in conformity with their dutie
  [Law 6 | labels cut: CORNER KICK GOAL KICK OFFSIDE PENALTY ...]
      was: • communicating information about a VAR-related decision to relevant parties CORNER KICK GOAL KICK O
  [Law 6 | diagram labels only]
      was: Refer to graphics for approved assistant referee signals. CORNER KICK GOAL KICK OFFSIDE PENALTY KICK
  [Law 6 | diagram labels only]
      was: FREE KICKFREE KICK Offside in the middle of the field Offside 

In [5]:
# The cleaned corpus is not saved to disk: it is rebuilt from the PDF in about a second.
# Instead, look at a sample to check the paragraphs and headings came out right.
print("\n--- Law 11 (offside), first lines of the rebuilt text ---")
for paragraph in corpus["Law 11"][:8]:
    print(("[heading] " if paragraph["kind"] == "heading" else "    ") + paragraph["text"][:150])


--- Law 11 (offside), first lines of the rebuilt text ---
[heading] 1. Offside position
    It is not an offence to be in an offside position. A player is in an offside position if:
    • any part of the head, body or feet is in the opponents’ half (excluding the halfway line) and
    • any part of the head, body or feet is nearer to the opponents’ goal line than both the ball and the second-last opponent
    The hands and arms of all players, including the goalkeepers, are not considered. For the purposes of determining offside, the upper boundary of the a
    • second-last opponent or
    • last two opponents
[heading] 2. Offside offence


## Stage 2 — Chunks

A **chunk** is the unit the retriever finds, so its size matters:

- **Too long:** the Bi-Encoder reads at most 256 tokens, and anything beyond that is silently cut.
  Long chunks also mix several rules into one vector.
- **Too short:** a clause like "unless the referee allows it" loses the rule it modifies.

The rules used here:

1. Never cross a Law boundary. A chunk never mixes Law 11 with Law 12.
2. Respect the numbered headings (`2. Offside offence`). A chunk stays within one heading.
3. Pack whole paragraphs and bullets up to `max_words`. Split only a paragraph that is longer than that,
   at a sentence boundary.
4. If a paragraph ends with a colon (a lead-in like "A player is in an offside position if:"), keep it
   with the bullets that follow it.
5. Merge a tiny last chunk (under `min_words`) into the one before it.

Each chunk also records its **context** (`Law 11 — Offside | 2. Offside offence`), which the next
stage puts in front of the text when it embeds the chunk.

In [6]:
max_words_per_chunk = config["chunking"]["max_words"]    # a chunk never goes above this
min_words_per_chunk = config["chunking"]["min_words"]    # a last chunk below this is merged into the one before

def split_long(text, max_words):
    """Cut one over-long paragraph into pieces of at most max_words, breaking at the end of a sentence."""
    pieces, buffer_words = [], []
    for sentence in re.split(r"(?<=[.!?])\s+", text):
        sentence_words = sentence.split()
        while len(sentence_words) > max_words:             # one huge sentence on its own: cut it by words
            if buffer_words:
                pieces.append(" ".join(buffer_words)); buffer_words = []
            pieces.append(" ".join(sentence_words[:max_words])); sentence_words = sentence_words[max_words:]
        if len(buffer_words) + len(sentence_words) > max_words and buffer_words:
            pieces.append(" ".join(buffer_words)); buffer_words = []   # this sentence would overflow, so close the piece
        buffer_words += sentence_words
    if buffer_words:
        pieces.append(" ".join(buffer_words))
    return pieces

def pack(paragraphs, max_words):
    """Pack whole paragraphs into chunks of at most max_words. Each chunk is a list of {"text", "pdf_page"}."""
    finished_chunks, current_chunk, current_word_count = [], [], 0
    for paragraph in paragraphs:
        pieces = split_long(paragraph["text"], max_words) if count_words(paragraph["text"]) > max_words else [paragraph["text"]]
        for piece in pieces:
            piece_words = count_words(piece)
            if current_word_count + piece_words > max_words and current_chunk:
                # The chunk is full. If it ends with a lead-in ("... if:"), move that lead-in to the
                # next chunk so it stays together with the bullets that follow it.
                lead_in = None
                if len(current_chunk) > 1 and current_chunk[-1]["text"].rstrip().endswith(":"):
                    lead_in = current_chunk.pop()
                finished_chunks.append(current_chunk)
                current_chunk, current_word_count = ([lead_in], count_words(lead_in["text"])) if lead_in else ([], 0)
            current_chunk.append({"text": piece, "pdf_page": paragraph["pdf_page"]})
            current_word_count += piece_words
    if current_chunk:
        finished_chunks.append(current_chunk)
    # A tiny last chunk joins the one before it, if the result is not much over the limit.
    if len(finished_chunks) > 1:
        last_chunk_words = sum(count_words(part["text"]) for part in finished_chunks[-1])
        last_two_words = sum(count_words(part["text"]) for chunk in finished_chunks[-2:] for part in chunk)
        if last_chunk_words < min_words_per_chunk and last_two_words <= max_words * 1.3:
            tiny_last_chunk = finished_chunks.pop()
            finished_chunks[-1] += tiny_last_chunk
    return finished_chunks

def section_title(section_name):
    """'Law 11' -> 'Law 11 - Offside'. Other sections (like 'VAR protocol') keep their own name."""
    law_match = re.fullmatch(r"Law (\d+)", section_name)
    return f"{section_name} — {LAW_TITLES[int(law_match.group(1))]}" if law_match else section_name

chunk_records = []
for section_name, paragraphs in corpus.items():
    # First split the section into blocks, one per numbered heading, so a chunk never crosses a heading.
    heading, block = "", []
    blocks = []                                            # a list of (heading, [paragraphs])
    for paragraph in paragraphs:
        if paragraph["kind"] == "heading":
            if block: blocks.append((heading, block))
            heading, block = paragraph["text"], []
        else:
            block.append(paragraph)
    if block: blocks.append((heading, block))
    # Then pack each block into chunks and record them.
    for heading, block in blocks:
        for chunk_parts in pack(block, max_words_per_chunk):
            chunk_text = "\n".join(part["text"] for part in chunk_parts)
            chunk_records.append({
                "id": f"c{len(chunk_records):04d}",        # c0000, c0001, ... in reading order
                "section": section_name,
                "context": section_title(section_name) + (f" | {heading}" if heading else ""),
                "heading": heading,
                "text": chunk_text,
                "pdf_page_start": chunk_parts[0]["pdf_page"],
                "pdf_page_end": chunk_parts[-1]["pdf_page"],
                "n_words": count_words(chunk_text),
            })
print("chunks:", len(chunk_records))

chunks: 230


In [7]:
# Sanity checks, then save. The chunk ids are what the hand-made queries point at, so the file is kept frozen.
assert len({chunk["id"] for chunk in chunk_records}) == len(chunk_records)       # every id is unique
assert all(chunk["n_words"] > 0 for chunk in chunk_records)                      # no empty chunk
corpus_words = sum(count_words(paragraph["text"]) for paragraphs in corpus.values() for paragraph in paragraphs if paragraph["kind"] == "para")
chunk_words = sum(chunk["n_words"] for chunk in chunk_records)
assert corpus_words == chunk_words, (corpus_words, chunk_words)                  # no text lost or duplicated

with open(project_root / "data" / "chunks.jsonl", "w", encoding="utf-8") as chunks_file:
    for chunk in chunk_records:
        chunks_file.write(json.dumps(chunk, ensure_ascii=False) + "\n")

chunk_sizes = sorted(chunk["n_words"] for chunk in chunk_records)
print(f"words in corpus = words in chunks = {chunk_words}")
print(f"chunk size (words): min {chunk_sizes[0]}, median {chunk_sizes[len(chunk_sizes)//2]}, max {chunk_sizes[-1]}")
print(f"chunks under {min_words_per_chunk} words: {sum(size < min_words_per_chunk for size in chunk_sizes)}")
print("\nchunks per section:")
for section_name in corpus:
    print(f"  {section_name:<42}{sum(chunk['section'] == section_name for chunk in chunk_records):>4}")

words in corpus = words in chunks = 26693
chunk size (words): min 13, median 131, max 187
chunks under 40 words: 13

chunks per section:
  Notes and modifications                     20
  Law 1                                       19
  Law 2                                        4
  Law 3                                       16
  Law 4                                       11
  Law 5                                       15
  Law 6                                        7
  Law 7                                        5
  Law 8                                        4
  Law 9                                        2
  Law 10                                       8
  Law 11                                       9
  Law 12                                      28
  Law 13                                       6
  Law 14                                       9
  Law 15                                       4
  Law 16                                       4
  Law 17                      

In [8]:
# Does the Bi-Encoder read every chunk in full? Count tokens (small word pieces) with the model's own tokenizer.
# A model silently ignores anything after its limit, so a chunk over the limit would lose its ending.
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/" + config["bi_encoder"])
bi_encoder_token_limit = 256                               # all-MiniLM-L6-v2 truncates after 256 tokens
token_counts = [len(tokenizer(chunk["context"] + "\n" + chunk["text"])["input_ids"]) for chunk in chunk_records]
print(f"tokens per chunk (context + text): median {sorted(token_counts)[len(token_counts)//2]}, max {max(token_counts)}")
print(f"chunks longer than {bi_encoder_token_limit}: {sum(count > bi_encoder_token_limit for count in token_counts)} of {len(token_counts)}")

tokens per chunk (context + text): median 169, max 258
chunks longer than 256: 1 of 230


In [9]:
# Look at a few chunks with your own eyes. This is the most useful check of all.
law_11_chunks = [chunk for chunk in chunk_records if chunk["section"] == "Law 11"]
law_12_chunks = [chunk for chunk in chunk_records if chunk["section"] == "Law 12"]
for chunk in law_11_chunks[:2] + law_12_chunks[5:6]:
    print(f"[{chunk['id']}] {chunk['context']}   (pdf p.{chunk['pdf_page_start']}, {chunk['n_words']} words)")
    print(chunk["text"])
    print("-" * 80)

[c0111] Law 11 — Offside | 1. Offside position   (pdf p.50, 116 words)
It is not an offence to be in an offside position. A player is in an offside position if:
• any part of the head, body or feet is in the opponents’ half (excluding the halfway line) and
• any part of the head, body or feet is nearer to the opponents’ goal line than both the ball and the second-last opponent
The hands and arms of all players, including the goalkeepers, are not considered. For the purposes of determining offside, the upper boundary of the arm is in line with the bottom of the armpit. A player is not in an offside position if level with the:
• second-last opponent or
• last two opponents
--------------------------------------------------------------------------------
[c0112] Law 11 — Offside | 2. Offside offence   (pdf p.50, 134 words)
A player in an offside position at the moment the ball is played or touched* by a team-mate is only penalised on becoming involved in active play by:
• interfering with 

## Stage 3 — Embed the chunks and build the index

The **Bi-Encoder** reads each chunk on its own and turns it into one vector (384 numbers). A question is
turned into a vector the same way, and the chunks whose vectors point in the most similar direction are the
retrieved ones. Because the vectors are normalised to length 1, the dot product *is* the cosine similarity,
so a plain inner-product FAISS index does exact cosine search.

Each chunk is embedded as `context + "\n" + text`, so the model also sees which Law it belongs to
(for example `Law 11`). This is cheap to recompute (230 chunks), so nothing is saved: the vectors are
rebuilt every run. Everything runs on the CPU, so the later timings do not depend on a busy GPU.

In [10]:
import numpy as np, faiss, time
from sentence_transformers import SentenceTransformer, CrossEncoder

# Load the frozen chunks from disk (the same ones every run).
chunks = [json.loads(line) for line in (project_root / "data" / "chunks.jsonl").read_text(encoding="utf-8").splitlines()]
chunk_ids = [chunk["id"] for chunk in chunks]
chunk_by_id = {chunk["id"]: chunk for chunk in chunks}
# The text the models actually read: the Law/heading label first, then the chunk text.
chunk_texts = [chunk["context"] + "\n" + chunk["text"] for chunk in chunks]

# Turn every chunk into a vector. normalize_embeddings=True makes each vector length 1, which
# makes the dot product equal to the cosine similarity.
bi_encoder = SentenceTransformer(config["bi_encoder"], device="cpu")
start_time = time.perf_counter()
chunk_vectors = bi_encoder.encode(chunk_texts, batch_size=32, normalize_embeddings=True,
                                  convert_to_numpy=True, show_progress_bar=False).astype("float32")
embedding_seconds = time.perf_counter() - start_time

# FAISS "IndexFlatIP" = exact search by inner product (no approximation, fine for 230 chunks).
faiss_index = faiss.IndexFlatIP(chunk_vectors.shape[1])
faiss_index.add(chunk_vectors)
print(f"{len(chunks)} chunks -> vectors {chunk_vectors.shape}, embedded in {embedding_seconds:.1f}s")
print("vector length (should be 1):", float(np.linalg.norm(chunk_vectors[0])))
print("index size:", faiss_index.ntotal)

230 chunks -> vectors (230, 384), embedded in 1.9s
vector length (should be 1): 1.0
index size: 230


## Stage 4 — Pipeline 1: Bi-Encoder retrieval (the baseline)

`retrieve(question)` embeds the question and asks the index for the **top 10** chunks
(`top_k_candidates`). The baseline sends the first 3 of them (`top_n_context`) to the LLM. Positions 4 to 10
are exactly the chunks the Cross-Encoder gets a chance to rescue.

In [11]:
num_candidates = config["retrieval"]["top_k_candidates"]     # how many chunks the Bi-Encoder returns (10)
num_context_chunks = config["retrieval"]["top_n_context"]    # how many chunks are pasted into the LLM prompt (3)

def retrieve(question, how_many=num_candidates):
    """Pipeline 1, step 1: the top chunks by cosine similarity, as a list of (chunk_id, bi_score), best first."""
    question_vector = bi_encoder.encode([question], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, positions = faiss_index.search(question_vector, how_many)
    return [(chunk_ids[position], float(score)) for position, score in zip(positions[0], scores[0])]

def baseline_context(question):
    """The ids of the 3 chunks the baseline sends to the LLM: simply the top 3 of the 10."""
    return [chunk_id for chunk_id, _ in retrieve(question)[:num_context_chunks]]

## Stage 5 — Pipeline 2: Cross-Encoder re-ranking

The **Cross-Encoder** reads the question and one chunk *together* in a single pass and outputs one relevance
score. It is slower than the Bi-Encoder (one pass per pair, 10 pairs per question), but it can compare the
words of the question against the words of the chunk, which the Bi-Encoder cannot. We run it only on the
same 10 candidates, re-sort them by its score and keep the top 3. The score is a raw model output, not a
probability; only the order matters.

In [12]:
cross_encoder = CrossEncoder(config["cross_encoder"], device="cpu", max_length=512)

def rerank(question, candidates):
    """Pipeline 2: re-score the candidates (a list of (chunk_id, bi_score)) with the Cross-Encoder
    and sort them, best first. Returns a list of (chunk_id, cross_score)."""
    question_chunk_pairs = [(question, chunk_texts[chunk_ids.index(chunk_id)]) for chunk_id, _ in candidates]
    cross_scores = cross_encoder.predict(question_chunk_pairs, show_progress_bar=False)
    scored = [(chunk_id, float(score)) for (chunk_id, _), score in zip(candidates, cross_scores)]
    return sorted(scored, key=lambda pair: -pair[1])

def reranked_context(question):
    """The ids of the 3 chunks the re-ranking pipeline sends to the LLM."""
    return [chunk_id for chunk_id, _ in rerank(question, retrieve(question))[:num_context_chunks]]

In [13]:
# A look at both rankings for one question. Not a test set: just a check that the two pipelines run.
def show(question):
    candidates = retrieve(question)                        # the baseline top 10
    reranked = rerank(question, candidates)                # the same 10, re-sorted
    old_rank = {chunk_id: rank for rank, (chunk_id, _) in enumerate(candidates, 1)}
    bi_score_of = dict(candidates)
    print("Q:", question)
    print(f"{'new':>3} {'old':>3}  {'chunk':<7}{'bi':>7}{'cross':>8}  section / heading")
    for new_rank, (chunk_id, cross_score) in enumerate(reranked, 1):
        chunk = chunk_by_id[chunk_id]
        # "promoted" = the chunk was outside the top 3 before but is inside it now
        mark = "  <- promoted into top 3" if new_rank <= num_context_chunks < old_rank[chunk_id] else ""
        print(f"{new_rank:>3} {old_rank[chunk_id]:>3}  {chunk_id:<7}{bi_score_of[chunk_id]:>7.3f}{cross_score:>8.2f}  {chunk['section']} / {chunk['heading'] or '-'}{mark}")

show("Can a player wear a ring or a bracelet during a match?")

Q: Can a player wear a ring or a bracelet during a match?
new old  chunk       bi   cross  section / heading
  1   1  c0059    0.631    6.35  Law 4 / 1. Safety
  2   3  c0062    0.465   -2.52  Law 4 / 3. Colours
  3   4  c0084    0.465   -3.09  Law 5 / 7. Liability of match officials  <- promoted into top 3
  4   9  c0064    0.414   -3.90  Law 4 / 4. Other equipment
  5   2  c0082    0.497   -4.08  Law 5 / 5. Referee’s equipment
  6   8  c0066    0.416   -4.22  Law 4 / 5. Slogans, statements, images and advertising
  7   7  c0063    0.416   -5.10  Law 4 / 4. Other equipment
  8   5  c0060    0.445   -5.20  Law 4 / 2. Compulsory equipment
  9   6  c0083    0.419   -5.67  Law 5 / 7. Liability of match officials
 10  10  c0135    0.402   -6.93  Law 12 / 3. Disciplinary action


## Stage 6 — Test queries and where the gold chunk lands

`data/queries.json` holds the hand-picked questions. Each has a **gold chunk** (the chunk that contains the
answer) and a short gold answer, both read off the chunk text. Four are queries where the Bi-Encoder
was expected to struggle (a keyword trap, an exception, a negation and a vocabulary gap) and two are
controls where it should do fine. Three more were added afterwards, and they did not all behave as
expected: **q7** (an injured player and cards) is a miss for both pipelines, **q8** (a water bottle) has no
gold chunk at all because the Laws do not cover it, so the right answer is a refusal, and **q9** (the
goalkeeper in a penalty shoot-out) is a vocabulary gap like q4. A query with no gold chunk cannot be ranked,
so it is left out of the Recall and MRR numbers.

For each question the table shows the rank of the gold chunk under each pipeline. A rank of 1 to 3 means
the LLM will see it. "Promoted" means rank 4 to 10 for the baseline but 1 to 3 after re-ranking.

In [14]:
import pandas as pd

queries = json.loads((project_root / "data" / "queries.json").read_text(encoding="utf-8"))

def best_rank(ranked_chunk_ids, gold_chunk_ids):
    """Position (1 = best) of the best-placed gold chunk in a ranked list of chunk ids, or None if it is not there."""
    ranks = [ranked_chunk_ids.index(gold) + 1 for gold in gold_chunk_ids if gold in ranked_chunk_ids]
    return min(ranks) if ranks else None

rankings = {}                                              # query id -> {"baseline": ..., "reranked": ...}
rank_rows = []
for query in queries:
    candidates = retrieve(query["question"])                       # Pipeline 1: Bi-Encoder top 10
    reranked = rerank(query["question"], candidates)               # Pipeline 2: same 10, re-sorted by the Cross-Encoder
    baseline_order = [chunk_id for chunk_id, _ in candidates]
    reranked_order = [chunk_id for chunk_id, _ in reranked]
    rankings[query["id"]] = {"baseline": candidates, "reranked": reranked}
    baseline_rank = best_rank(baseline_order, query["gold_chunks"])
    reranked_rank = best_rank(reranked_order, query["gold_chunks"])
    rank_rows.append({
        "id": query["id"], "category": query["category"], "gold": ",".join(query["gold_chunks"]),
        "bi_rank": baseline_rank, "ce_rank": reranked_rank,        # column names kept short: they are the saved CSV's format
        "in_top3_baseline": bool(baseline_rank and baseline_rank <= num_context_chunks),
        "in_top3_rerank": bool(reranked_rank and reranked_rank <= num_context_chunks),
        # promoted = outside the top 3 for the baseline, inside it after re-ranking
        "promoted": bool(baseline_rank and reranked_rank and baseline_rank > num_context_chunks and reranked_rank <= num_context_chunks),
    })
rank_table = pd.DataFrame(rank_rows)
rank_table[["bi_rank", "ce_rank"]] = rank_table[["bi_rank", "ce_rank"]].astype("Int64")   # whole numbers; blank when a query has no gold chunk
(project_root / "results").mkdir(exist_ok=True)
rank_table.to_csv(project_root / "results" / "rank_shifts.csv", index=False)
rank_table

,id,category,gold,bi_rank,ce_rank,in_top3_baseline,in_top3_rerank,promoted
0,q1,keyword trap,c0099,5,1,False,True,True
1,q2,exception,c0138,9,1,False,True,True
2,q3,negation,c0111,8,1,False,True,True
3,q4,vocabulary gap,c0041,6,1,False,True,True
4,q5,control,c0059,1,1,True,True,False
5,q6,control,c0043,1,1,True,True,False
6,q7,other,c0077,6,5,False,False,False
7,q8,not covered,,<NA>,<NA>,False,False,False
8,q9,vocabulary gap,c0108,10,2,False,True,True
9,q10,partly covered,c0043,2,1,True,True,False


In [15]:
# The full top 10 for each query: old position (baseline) -> new position (re-ranked), with both scores.
for query in queries:
    candidates = rankings[query["id"]]["baseline"]
    reranked = rankings[query["id"]]["reranked"]
    old_rank = {chunk_id: rank for rank, (chunk_id, _) in enumerate(candidates, 1)}
    bi_score_of = dict(candidates)
    print(f"{query['id']} [{query['category']}] {query['question']}   gold: {query['gold_chunks']}")
    print(f"  {'new':>3} {'old':>3}  {'chunk':<7}{'bi':>7}{'cross':>8}  section / heading")
    for new_rank, (chunk_id, cross_score) in enumerate(reranked, 1):
        chunk = chunk_by_id[chunk_id]
        flag = "  <- GOLD" if chunk_id in query["gold_chunks"] else ""
        print(f"  {new_rank:>3} {old_rank[chunk_id]:>3}  {chunk_id:<7}{bi_score_of[chunk_id]:>7.3f}{cross_score:>8.2f}  {chunk['section']} / {chunk['heading'] or '-'}{flag}")
    print()

q1 [keyword trap] Can a goal be scored directly from a kick-off?   gold: ['c0099']
  new old  chunk       bi   cross  section / heading
    1   5  c0099    0.623    9.60  Law 8 / 1. Kick-off  <- GOLD
    2   1  c0167    0.690    8.49  Law 16 / -
    3   4  c0154    0.630    7.79  Law 14 / -
    4   6  c0171    0.617    7.61  Law 17 / -
    5   2  c0103    0.655    5.67  Law 10 / 1. Goal scored
    6   7  c0159    0.609    2.40  Law 14 / 2. Offences and sanctions
    7   3  c0170    0.648    1.48  Law 16 / 2. Offences and sanctions
    8   9  c0169    0.598    0.02  Law 16 / 2. Offences and sanctions
    9  10  c0119    0.595   -1.10  Law 11 / 4. Offences and sanctions
   10   8  c0105    0.608   -2.92  Law 10 / 3. Penalties (penalty shoot-out)

q2 [exception] Is a goalkeeper sent off for handling the ball inside their own penalty area to stop a goal?   gold: ['c0138']
  new old  chunk       bi   cross  section / heading
    1   9  c0138    0.628    6.09  Law 12 / 3. Disciplinary action

### Summary metrics: Recall@3 and MRR

Two numbers over the table above, one per pipeline.

- **Recall@3**: the fraction of queries whose gold chunk is in the top 3, i.e. the chunks the LLM actually sees.
- **MRR** (mean reciprocal rank): the average of 1 / rank of the gold chunk. Rank 1 scores 1.0, rank 5 scores 0.2, rank 9 scores 0.11.

> These are a demonstration, not an evaluation. The first four queries were **chosen because the
> baseline fails on them**, so they favour re-ranking by construction. The two controls show it does
> not hurt when the baseline is already right. With six queries, one query moves a number a lot.


In [16]:
def recall_at_n(gold_ranks, n=num_context_chunks):
    """Fraction of queries whose gold chunk is within the top n. A gold chunk that is not in the top 10 counts as a miss."""
    return sum(1 for rank in gold_ranks if pd.notna(rank) and rank <= n) / len(gold_ranks)

def mean_reciprocal_rank(gold_ranks):
    """Average of 1 / rank. Rank 1 gives 1.0, rank 5 gives 0.2. A gold chunk missing from the top 10 gives 0."""
    return sum(1 / rank if pd.notna(rank) else 0 for rank in gold_ranks) / len(gold_ranks)

def summarise(table):
    """The four numbers for one set of queries."""
    return {"Recall@3, baseline":  recall_at_n(table["bi_rank"]),  "Recall@3, re-ranked": recall_at_n(table["ce_rank"]),
            "MRR, baseline":       mean_reciprocal_rank(table["bi_rank"]), "MRR, re-ranked": mean_reciprocal_rank(table["ce_rank"])}

answerable_table = rank_table[rank_table["category"] != "not covered"]    # a query with no gold chunk cannot be ranked
test_table = answerable_table[answerable_table["category"] != "control"]    # leave out the controls
metrics = pd.DataFrame({f"All {len(answerable_table)} answerable queries": summarise(answerable_table),
                        f"{len(test_table)} non-control queries only": summarise(test_table)}).round(3)
metrics.to_csv(project_root / "results" / "retrieval_metrics.csv")
metrics

,All 14 answerable queries,12 non-control queries only
"Recall@3, baseline",0.429,0.333
"Recall@3, re-ranked",0.786,0.750
"MRR, baseline",0.419,0.322
"MRR, re-ranked",0.693,0.642


## Stage 7 — Answer 1 and Answer 2 (Gemini)

Both pipelines use the **same LLM, the same settings and the same prompt**; only the three chunks pasted into
the prompt differ. The prompt, verbatim:

```
Answer the question using only the context below, which is taken from the IFAB Laws of the Game 2024/25.
If the context does not contain the answer, reply exactly: The context does not contain the answer.
Answer in at most three sentences.

Context:
[1] <section / heading>
<chunk text>

[2] <section / heading>
<chunk text>

[3] <section / heading>
<chunk text>

Question: <question>
Answer:
```

**Controls on the API.** Calls are made one at a time, at least `min_seconds_between_calls` apart. Only
rate-limit and server errors (429, 5xx) are retried, with growing waits (up to 60 s); any other error stops at once.
Each answer is capped at `max_output_tokens`. The key is read from `.env` and never printed.

Answers are saved to `results/answers.json` after every call, so a crash loses nothing. With `USE_SAVED_ANSWERS = True` an answer whose prompt and model have not
changed is reused, so re-running the notebook makes no API calls. Set it to `False` to ask again.

In [17]:
import os, hashlib
from dotenv import load_dotenv

USE_SAVED_ANSWERS = True                                   # True = reuse answers from disk, so no API calls are made
answers_path = project_root / "results" / "answers.json"
llm_config = config["llm"]

# The instruction placed above the context. It is the SAME for both pipelines.
PROMPT_HEAD = (
    "Answer the question using only the context below, which is taken from the IFAB Laws of the Game 2024/25.\n"
    "If the context does not contain the answer, reply exactly: The context does not contain the answer.\n"
    "Answer in at most three sentences.\n\nContext:\n"
)

def build_prompt(question, context_chunk_ids):
    """Put the instruction, the numbered chunks and the question together into one prompt."""
    numbered_chunks = []
    for number, chunk_id in enumerate(context_chunk_ids, 1):
        chunk = chunk_by_id[chunk_id]
        numbered_chunks.append(f"[{number}] {chunk['context']}\n{chunk['text']}")
    return PROMPT_HEAD + "\n\n".join(numbered_chunks) + f"\n\nQuestion: {question}\nAnswer:"

saved_answers = json.loads(answers_path.read_text(encoding="utf-8")) if answers_path.exists() else {}

gemini_client = None                                       # created on the first real API call
last_call_time = [0.0]                                     # when the previous call started (for the throttle)

def ask_llm(prompt):
    """Make one Gemini call. Returns (answer text, seconds spent inside the successful call)."""
    global gemini_client
    if gemini_client is None:
        from google import genai
        from google.genai import types
        load_dotenv(project_root / ".env")
        api_key = os.environ.get(llm_config["api_key_env"], "").strip()
        assert api_key, f"No {llm_config['api_key_env']} found. Put it in repo/.env"
        # A 60-second limit per call, so a busy or rate-limited API cannot make the notebook wait for minutes.
        gemini_client = genai.Client(api_key=api_key, http_options=types.HttpOptions(timeout=60000))
    from google.genai import types
    for attempt in range(llm_config["max_retries"] + 1):
        # Throttle: wait so that calls are at least min_seconds_between_calls apart (free tier has a low limit).
        wait_seconds = llm_config["min_seconds_between_calls"] - (time.time() - last_call_time[0])
        if wait_seconds > 0:
            time.sleep(wait_seconds)
        last_call_time[0] = time.time()
        start_time = time.perf_counter()
        try:
            response = gemini_client.models.generate_content(
                model=llm_config["model"], contents=prompt,
                config=types.GenerateContentConfig(temperature=llm_config["temperature"],
                                                   max_output_tokens=llm_config["max_output_tokens"]))
            elapsed_seconds = time.perf_counter() - start_time
            finish_reason = str(response.candidates[0].finish_reason) if response.candidates else "NO_CANDIDATE"
            # Refuse a cut-off answer (for example MAX_TOKENS) instead of silently saving half an answer.
            assert response.text and finish_reason.endswith("STOP"), f"answer not complete: {finish_reason}"
            return response.text.strip(), elapsed_seconds
        except Exception as error:
            # Retry only "too many requests" and server errors; anything else is a real bug, so stop at once.
            if getattr(error, "code", None) not in (429, 500, 502, 503, 504) or attempt == llm_config["max_retries"]:
                raise
            time.sleep(min(2 ** (attempt + 2), 60))        # wait 4, 8, 16, 32, 60 ... seconds before trying again

answers = {}
api_calls_made = 0
for query in queries:
    for pipeline, ranked in (("baseline", rankings[query["id"]]["baseline"]), ("rerank", rankings[query["id"]]["reranked"])):
        context_ids = [chunk_id for chunk_id, _ in ranked[:num_context_chunks]]     # the top 3 for this pipeline
        prompt = build_prompt(query["question"], context_ids)
        prompt_hash = hashlib.sha256(prompt.encode("utf-8")).hexdigest()[:12]       # fingerprint: has the prompt changed?
        answer_key = f"{query['id']}|{pipeline}"
        already_saved = (answer_key in saved_answers and saved_answers[answer_key]["prompt_sha"] == prompt_hash
                         and saved_answers[answer_key].get("model") == llm_config["model"])
        if USE_SAVED_ANSWERS and already_saved:
            answers[answer_key] = saved_answers[answer_key]
            continue
        answer_text, llm_seconds = ask_llm(prompt)
        api_calls_made += 1
        answers[answer_key] = {"qid": query["id"], "pipeline": pipeline, "chunks": context_ids, "prompt_sha": prompt_hash,
                               "answer": answer_text, "llm_seconds": round(llm_seconds, 2), "model": llm_config["model"]}
        # Save after every call, so a crash later does not lose answers already paid for.
        answers_path.write_text(json.dumps({**saved_answers, **answers}, indent=2, ensure_ascii=False), encoding="utf-8")

answers_path.write_text(json.dumps(answers, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"{len(answers)} answers ready ({api_calls_made} new API calls, {len(answers) - api_calls_made} reused from disk)")

30 answers ready (0 new API calls, 30 reused from disk)


In [18]:
# Answer 1 (baseline) next to Answer 2 (re-ranked), with the gold answer to compare against.
for query in queries:
    baseline_answer = answers[f"{query['id']}|baseline"]
    reranked_answer = answers[f"{query['id']}|rerank"]
    print(f"{query['id']} [{query['category']}] {query['question']}")
    print(f"  gold chunk {query['gold_chunks']}  | baseline sent {baseline_answer['chunks']}  | rerank sent {reranked_answer['chunks']}")
    print(f"  GOLD     : {query['gold_answer']}")
    print(f"  ANSWER 1 : {baseline_answer['answer']}")
    print(f"  ANSWER 2 : {reranked_answer['answer']}")
    print()

q1 [keyword trap] Can a goal be scored directly from a kick-off?
  gold chunk ['c0099']  | baseline sent ['c0167', 'c0103', 'c0170']  | rerank sent ['c0099', 'c0167', 'c0154']
  GOLD     : Yes, against the opponents. If the ball goes directly into the kicker's own goal, a corner kick is awarded to the opponents.
  ANSWER 1 : The context does not contain the answer.
  ANSWER 2 : Yes, a goal may be scored directly against the opponents from the kick-off. However, if the ball directly enters the kicker’s goal, a corner kick is awarded to the opponents.

q2 [exception] Is a goalkeeper sent off for handling the ball inside their own penalty area to stop a goal?
  gold chunk ['c0138']  | baseline sent ['c0155', 'c0134', 'c0107']  | rerank sent ['c0138', 'c0126', 'c0155']
  GOLD     : No. A deliberate handball that denies a goal or an obvious goal-scoring opportunity is a sending-off wherever it happens, except for a goalkeeper within their own penalty area.
  ANSWER 1 : The context does not 

## Stage 8 — Latency

Timed **one step at a time, never in parallel**, after a warm-up pass (the first call to a model is slower
because of lazy initialisation). Each retrieval step is timed over every query, 5 repeats, and the table
reports the **median**. Both encoders run on the CPU here.

The LLM is a network call to Google, so its time includes the internet and Google's load. It is the time
of the successful call only, without the throttle waits or retries, and it is **not comparable** with the
local steps. It was measured when the answers were first generated (it is stored in `answers.json`), so
if answers were reused from disk the figure is from that earlier run.

In [19]:
import statistics

REPEATS = 5                                                # time every query this many times, then take the median
questions = [query["question"] for query in queries]

# Warm-up: one throw-away pass, so the slow first call (lazy start-up) is not counted.
for question in questions[:2]:
    rerank(question, retrieve(question))

# Time the two local steps one after the other, never in parallel.
retrieval_ms_list, rerank_ms_list = [], []
for question in questions:
    for _ in range(REPEATS):
        start = time.perf_counter(); candidates = retrieve(question); after_retrieval = time.perf_counter()
        rerank(question, candidates); after_rerank = time.perf_counter()
        retrieval_ms_list.append((after_retrieval - start) * 1000)
        rerank_ms_list.append((after_rerank - after_retrieval) * 1000)

# The LLM time was stored in answers.json when each answer was first generated.
llm_seconds_by_pipeline = {pipeline: [answer["llm_seconds"] for answer in answers.values() if answer["pipeline"] == pipeline]
                           for pipeline in ("baseline", "rerank")}
median = statistics.median
retrieval_ms, rerank_ms = median(retrieval_ms_list), median(rerank_ms_list)
latency = pd.DataFrame([
    {"step": "Bi-Encoder retrieval (embed question + search 230 chunks)", "baseline_ms": retrieval_ms, "rerank_ms": retrieval_ms},
    {"step": "Cross-Encoder re-ranking of the 10 candidates",            "baseline_ms": 0.0,          "rerank_ms": rerank_ms},
    {"step": "Retrieval total (local, CPU)",                             "baseline_ms": retrieval_ms, "rerank_ms": retrieval_ms + rerank_ms},
    {"step": "LLM call (Gemini API, median, network included)",
     "baseline_ms": median(llm_seconds_by_pipeline["baseline"]) * 1000, "rerank_ms": median(llm_seconds_by_pipeline["rerank"]) * 1000},
]).round(1)
latency.to_csv(project_root / "results" / "latency.csv", index=False)
print(f"median over {len(retrieval_ms_list)} timings ({len(questions)} queries x {REPEATS} repeats)")
print(f"re-ranking costs {rerank_ms / retrieval_ms:.1f}x the retrieval step on top of it")
latency

median over 75 timings (15 queries x 5 repeats)
re-ranking costs 11.6x the retrieval step on top of it


,step,baseline_ms,rerank_ms
0,Bi-Encoder retrieval (embed question + search ...,8.6,8.6
1,Cross-Encoder re-ranking of the 10 candidates,0.0,99.3
2,"Retrieval total (local, CPU)",8.6,107.8
3,"LLM call (Gemini API, median, network included)",4000.0,5130.0


## Stage 9 — Does the wording of a question matter? (rephrased queries)

For queries where nothing was retrieved, nothing changed, or the LLM refused, the question was rewritten in
other natural words (`data/rephrased_queries.json`, 16 rewrites). The **gold chunk is the same** as for the
original query; only the question's wording differs. For each rewrite the table shows the gold chunk's rank
before and after, and whether it changed.

> **Read with care.** The rewrites were written *after* seeing how the originals behaved, so this is a check on
> how sensitive the system is to wording, not an independent test. Every rewrite that was tried is listed,
> including the ones that changed nothing.

To stay inside the free API limit, the LLM is asked only for one chosen rewrite per original query (marked
`ask_llm` in the file). If the daily limit is reached, the remaining answers are skipped and shown as "not run".
These answers are saved in `results/answers_rephrased.json`, separate from the main answers.


In [20]:
rephrased_queries = json.loads((project_root / "data" / "rephrased_queries.json").read_text(encoding="utf-8"))
original_by_id = {query["id"]: query for query in queries}
original_rank_row = {row["id"]: row for row in rank_rows}      # the rank table of the original queries

rephrased_rows = []
rephrased_rankings = {}
for rewrite in rephrased_queries:
    gold_chunk_ids = original_by_id[rewrite["original"]]["gold_chunks"]        # same gold chunk as the original
    candidates = retrieve(rewrite["question"])
    reranked = rerank(rewrite["question"], candidates)
    rephrased_rankings[rewrite["id"]] = {"baseline": candidates, "reranked": reranked}
    baseline_rank = best_rank([chunk_id for chunk_id, _ in candidates], gold_chunk_ids)
    reranked_rank = best_rank([chunk_id for chunk_id, _ in reranked], gold_chunk_ids)
    original_baseline_rank = original_rank_row[rewrite["original"]]["bi_rank"]
    original_reranked_rank = original_rank_row[rewrite["original"]]["ce_rank"]
    rephrased_rows.append({
        "id": rewrite["id"], "original": rewrite["original"], "question": rewrite["question"],
        "original_rank_pipeline1": original_baseline_rank, "original_rank_pipeline2": original_reranked_rank,
        "new_rank_pipeline1": baseline_rank, "new_rank_pipeline2": reranked_rank,
        # "changed" = the gold chunk sits at a different rank in either pipeline than it did for the original wording
        "changed": (baseline_rank != original_baseline_rank) or (reranked_rank != original_reranked_rank),
    })
rephrased_table = pd.DataFrame(rephrased_rows)
for column in ["original_rank_pipeline1", "original_rank_pipeline2", "new_rank_pipeline1", "new_rank_pipeline2"]:
    rephrased_table[column] = rephrased_table[column].astype("Int64")      # whole numbers; blank = not in the top 10
rephrased_table.to_csv(project_root / "results" / "rephrased_rank_shifts.csv", index=False)
print(f"{len(rephrased_table)} rewrites tried, {int(rephrased_table['changed'].sum())} changed the gold chunk's rank")
rephrased_table

16 rewrites tried, 11 changed the gold chunk's rank


,id,original,question,original_rank_pipeline1,original_rank_pipeline2,new_rank_pipeline1,new_rank_pipeline2,changed
0,q7r1,q7,Can the referee caution or send off a player w...,6,5,6,1,True
1,q7r2,q7,Does an injured player get a card from the ref...,6,5,2,1,True
2,q7r3,q7,Can a player who is hurt and leaving the pitch...,6,5,4,3,True
3,q8r1,q8,Are players allowed to carry a bottle of water...,<NA>,<NA>,<NA>,<NA>,False
4,q9r1,q9,Can an outfield player take the goalkeeper's p...,10,2,<NA>,<NA>,True
5,q9r2,q9,Can the goalkeeper take a penalty kick in a sh...,10,2,6,2,True
6,q10r1,q10,Is there a limit on how many players of one te...,2,1,1,1,True
7,q10r2,q10,What happens if too many players of a team are...,2,1,1,3,True
8,q11r1,q11,Are team officials allowed on the field during...,1,1,2,1,True
9,q11r2,q11,Who is allowed on the pitch during a penalty s...,1,1,2,1,True


In [21]:
# Answers for the chosen rewrites only (to stay inside the free daily limit of the API).
rephrased_answers_path = project_root / "results" / "answers_rephrased.json"
saved_rephrased = json.loads(rephrased_answers_path.read_text(encoding="utf-8")) if rephrased_answers_path.exists() else {}
rephrased_answers = dict(saved_rephrased)
retries_before = llm_config["max_retries"]
llm_config["max_retries"] = 0                                # no retries here: if the API is busy, stop at once
for rewrite in rephrased_queries:
    if not rewrite["ask_llm"]:
        continue
    for pipeline, ranked in (("baseline", rephrased_rankings[rewrite["id"]]["baseline"]), ("rerank", rephrased_rankings[rewrite["id"]]["reranked"])):
        answer_key = f"{rewrite['id']}|{pipeline}"
        context_ids = [chunk_id for chunk_id, _ in ranked[:num_context_chunks]]
        prompt = build_prompt(rewrite["question"], context_ids)
        prompt_hash = hashlib.sha256(prompt.encode("utf-8")).hexdigest()[:12]
        if answer_key in rephrased_answers and rephrased_answers[answer_key]["prompt_sha"] == prompt_hash:
            continue                                         # already answered with this exact prompt
        try:
            answer_text, llm_seconds = ask_llm(prompt)
        except Exception as error:                           # most likely the free daily limit: stop asking, keep what we have
            print(f"stopped at {answer_key}: {str(error)[:120]}")
            break
        rephrased_answers[answer_key] = {"qid": rewrite["id"], "pipeline": pipeline, "chunks": context_ids, "prompt_sha": prompt_hash,
                                         "answer": answer_text, "llm_seconds": round(llm_seconds, 2), "model": llm_config["model"]}
        rephrased_answers_path.write_text(json.dumps(rephrased_answers, indent=2, ensure_ascii=False), encoding="utf-8")
    else:
        continue
    break                                                    # leave the outer loop too once the limit is hit
llm_config["max_retries"] = retries_before                  # put the setting back
print(f"{len(rephrased_answers)} answers for rewrites saved")

# Original answer next to the rewrite's answers, for the chosen rewrites.
for rewrite in rephrased_queries:
    if not rewrite["ask_llm"]:
        continue
    print(f"{rewrite['id']} (rewrite of {rewrite['original']}): {rewrite['question']}")
    for pipeline, label in (("baseline", "ANSWER 1"), ("rerank", "ANSWER 2")):
        saved = rephrased_answers.get(f"{rewrite['id']}|{pipeline}")
        print(f"  {label}: {saved['answer'] if saved else 'not run'}")
    print()

12 answers for rewrites saved
q7r1 (rewrite of q7): Can the referee caution or send off a player who is injured?
  ANSWER 1: The context does not contain the answer.
  ANSWER 2: Yes. The referee can caution or send off an injured player, but if the player has to leave the field of play for treatment, the card must be shown before the player leaves.

q9r2 (rewrite of q9): Can the goalkeeper take a penalty kick in a shoot-out?
  ANSWER 1: Yes, the context indicates that a goalkeeper can take a kick, noting that "if the goalkeeper has already taken a kick, the replacement may not take a kick until the next round of kicks." However, a goalkeeper who is unable to continue and is replaced takes no further part and may not take a kick.
  ANSWER 2: Yes, a goalkeeper can take a kick in a penalty shoot-out, as the context indicates that a goalkeeper may have already taken a kick before being replaced. However, a replaced goalkeeper who is unable to continue takes no further part and may not take